# Scraping Land Processing: limpieza y estandarización

Notebook para limpiar `scraping_land_raw.csv` con foco en nulos, tipado, estandarización de `tipo_suelo`, variables derivadas y deduplicación.

El tratamiento de outliers se realiza en el notebook `scraping_land_processing_outliers`.

**Trazabilidad:** cada acción genera un dataframe nuevo; no se reasigna `df`.

**Input:**  `data/raw/scraping_manual/raw/scraping_land_raw.csv`

**Output:** `data/raw/scraping_manual/preprocessed/scraping_land_preprocessed.csv`

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


## Carga del dataset

Se lee el CSV desde el repositorio para trabajar siempre con la versión local de esta rama.


In [ ]:
csv_candidates = [
    Path("../../data/raw/scraping_manual/raw/scraping_land_raw.csv"),
    Path("data/raw/scraping_manual/raw/scraping_land_raw.csv")
]
csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])

# Leemos el raw original (no se modifica nunca en disco).
df_raw_original = pd.read_csv(
    csv_path,
    delimiter=";",
    on_bad_lines="warn",
    encoding="utf-8",
    encoding_errors="replace",
    skipinitialspace=True,
    index_col=0
)

# Copia de trabajo: toda la limpieza del notebook se aplica sobre esta copia.
df_raw = df_raw_original.copy(deep=True)

print(f"CSV cargado desde: {csv_path}")
print(f"Filas: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")
df_raw.head(3)

CSV cargado desde: ..\..\data\raw\scraping_manual\raw\scraping_land_raw.csv
Filas: 1,299 | Columnas: 11


,anunciante,titulo,municipio,zona,precio_eur,precio_anterior_eur,descuento_pct,superficie_m2,tipo_suelo,referencia_catastral,notas
id,,,,,,,,,,,
1,Exclusivas inmobiliarias mikeli,"Terreno en Barrio Llanilla, Alisal - Cazoña - ...",Santander,"Barrio Llanilla, Alisal - Cazoña - San Román",430000,NaN,NaN,972,NaN,NaN,NaN
2,MENA INMOBILIARIA,"Terreno en Oruña, Piélagos",Piélagos,Oruña,139000,NaN,NaN,965,Urbano (solar),NaN,Edificabilidad 200 m²
3,Grupo Inmobiliario Tribeca,"Terreno en Peñacastillo - Nuevamontaña, Santander",Santander,Peñacastillo - Nuevamontaña,169000,NaN,NaN,650,Urbano (solar),NaN,5 parcelas


## Normalización inicial y tratamiento de nulos

1. Se limpia texto (espacios).
2. Se convierten marcadores vacíos a nulo real (`NaN`).
3. Se identifican registros con al menos una variable nula.


In [ ]:
# Estandarizar vacíos y marcadores de no dato en todo el dataframe.
missing_tokens = {
    "", " ", "na", "n/a", "none", "null", "nan", "-", "--", "sin dato", "no informado"
}

df_clean = df_raw.copy()

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace({token: np.nan for token in missing_tokens})

# Máscara de registros incompletos (alguna variable nula).
mask_registro_incompleto = df_clean.isna().any(axis=1)

print("Registros con al menos un nulo:", int(mask_registro_incompleto.sum()))
print("Porcentaje sobre total:", round(mask_registro_incompleto.mean() * 100, 2), "%")

df_clean.loc[mask_registro_incompleto].head(10)


Registros con al menos un nulo: 1297
Porcentaje sobre total: 99.85 %


,anunciante,titulo,municipio,zona,precio_eur,precio_anterior_eur,descuento_pct,superficie_m2,tipo_suelo,referencia_catastral,notas
id,,,,,,,,,,,
1,Exclusivas inmobiliarias mikeli,"Terreno en Barrio Llanilla, Alisal - Cazoña - ...",Santander,"Barrio Llanilla, Alisal - Cazoña - San Román",430000,NaN,NaN,972,NaN,NaN,NaN
2,MENA INMOBILIARIA,"Terreno en Oruña, Piélagos",Piélagos,Oruña,139000,NaN,NaN,965,Urbano (solar),NaN,Edificabilidad 200 m²
3,Grupo Inmobiliario Tribeca,"Terreno en Peñacastillo - Nuevamontaña, Santander",Santander,Peñacastillo - Nuevamontaña,169000,NaN,NaN,650,Urbano (solar),NaN,5 parcelas
4,Exclusivas inmobiliarias mikeli,"Terreno en Calle de la Arnia, Soto de la Marin...",Santa Cruz de Bezana,"Calle de la Arnia, Soto de la Marina",1290000,NaN,NaN,3217,NaN,NaN,Parcela urbana
5,Exclusivas inmobiliarias mikeli,"Terreno en Calle Barrio San Roque, Polanco",Polanco,Calle Barrio San Roque,80000,NaN,NaN,1668,NaN,NaN,820 urbanos + 848 rústicos
6,Exclusivas inmobiliarias mikeli,"Terreno en Avenida Playa San Juan de la Canal,...",Santa Cruz de Bezana,"Av. Playa San Juan de la Canal, Soto de la Marina",1500000,NaN,NaN,6518,NaN,NaN,3.433 urbanos
7,Bahía Home,"Terreno en Barrio la Venera, Oruña, Piélagos",Piélagos,"Barrio la Venera, Oruña",98000,NaN,NaN,4507,Urbanizable,NaN,Licencia + proyecto
8,Distrito inmobiliario,"Terreno en Barrio Mar, 87 --81, Polanco",Polanco,"Barrio Mar, 87 --81",132000,140000.0,6.0,2210,Urbano (solar),NaN,Licencia + proyecto básico
9,RE/MAX Vértice,"Terreno en Sv-4713, 12, Miengo",Miengo,"Sv-4713, 12",300000,NaN,NaN,6548,No urbanizable,NaN,Proyecto hotel canino


## Tipado y saneamiento de variables numéricas

Se fuerzan a numérico las variables cuantitativas; cualquier texto o formato inválido se transforma en nulo.


In [ ]:
def to_numeric_strict(series: pd.Series) -> pd.Series:
    # Convierte texto a número; si hay texto no numérico, deja NaN.
    cleaned = (
        series.astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .str.replace(r"[^0-9,.-]", "", regex=True)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")

df_typed = df_clean.copy()

numeric_cols = ["precio_eur", "precio_anterior_eur", "descuento_pct", "superficie_m2"]

print(df_typed[numeric_cols].dtypes)
df_typed[numeric_cols].describe(include="all").T


precio_eur               int64
precio_anterior_eur    float64
descuento_pct          float64
superficie_m2            int64
dtype: object


,count,mean,std,min,25%,50%,75%,max
precio_eur,1299.0,260523.380293,376253.654300,1200.0,80000.0,149000.0,280600.0,5000000.0
precio_anterior_eur,42.0,198264.285714,168272.857428,58900.0,92800.0,139000.0,200000.0,695000.0
descuento_pct,135.0,4.333333,9.622734,0.0,0.0,0.0,5.0,50.0
superficie_m2,1299.0,6254.883757,12252.832514,1.0,1291.5,2842.0,5827.5,150000.0


## Eliminación de columna no requerida

Se elimina la variable `zona` como pediste.


In [ ]:
df_nozona = df_typed.copy()

if "zona" in df_nozona.columns:
    df_nozona = df_nozona.drop(columns=["zona"])

print("Columnas actuales:", df_nozona.columns.tolist())


Columnas actuales: ['anunciante', 'titulo', 'municipio', 'precio_eur', 'precio_anterior_eur', 'descuento_pct', 'superficie_m2', 'tipo_suelo', 'referencia_catastral', 'notas']


## Variable booleana de descuento

- Si `descuento_pct` tiene valor: `vendido_con_descuento = True`.
- Si `descuento_pct` está nulo: `descuento_pct` queda nulo y `vendido_con_descuento = False` (no descuento identificado).


In [ ]:
df_descuento = df_nozona.copy()

df_descuento["vendido_con_descuento"] = df_descuento["descuento_pct"].notna()

df_descuento[["descuento_pct", "vendido_con_descuento"]].head(10)


,descuento_pct,vendido_con_descuento
id,,
1,NaN,False
2,NaN,False
3,NaN,False
4,NaN,False
5,NaN,False
6,NaN,False
7,NaN,False
8,6.0,True
9,NaN,False


## Limpieza de referencia catastral (string)

Se asegura tipo string y se convierte a nulo cuando no hay contenido real.


In [ ]:
df_catastro = df_descuento.copy()

df_catastro["referencia_catastral"] = (
    df_catastro["referencia_catastral"]
    .astype("string")
    .str.strip()
)
df_catastro.loc[df_catastro["referencia_catastral"].isin(["", "nan", "None", "null"]), "referencia_catastral"] = pd.NA

print("Nulos en referencia_catastral:", int(df_catastro["referencia_catastral"].isna().sum()))


Nulos en referencia_catastral: 1236


## Normalización de `tipo_suelo` con apoyo de `notas`

Reglas aplicadas:
1. `Urbano (según texto)` -> `Urbano (solar)`.
2. Cuando `tipo_suelo` es vacío o `No especificado`, se infiere desde `notas`.
3. Si en `notas` aparecen a la vez urbano y rústico, prevalece `Urbano (solar)`.
4. Si no hay patrón claro, queda nulo.


In [ ]:
def normalize_text(value: str) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    return "".join(ch for ch in value if not unicodedata.combining(ch))

df_tipo_suelo = df_catastro.copy()

# Normalización base de tipo_suelo.
df_tipo_suelo["tipo_suelo"] = df_tipo_suelo["tipo_suelo"].astype("string").str.strip()
df_tipo_suelo.loc[df_tipo_suelo["tipo_suelo"].isin(["", "No especificado", "no especificado", "nan"]), "tipo_suelo"] = pd.NA
df_tipo_suelo["tipo_suelo"] = df_tipo_suelo["tipo_suelo"].replace({"Urbano (según texto)": "Urbano (solar)"})

mask_infer = df_tipo_suelo["tipo_suelo"].isna()

def infer_tipo_suelo_from_notas(notas: str):
    t = normalize_text(notas)

    has_urbano = bool(re.search(r"\burban[oa]s?\b|parcela urbana|suelo urbano", t))
    has_urbanizable = "urbanizable" in t
    has_industrial = "industrial" in t
    has_edificable = bool(re.search(r"edificable|edificabilidad", t))
    has_rustico = bool(re.search(r"rustic", t))

    if has_urbano and has_rustico:
        return "Urbano (solar)"
    if has_urbano or has_edificable:
        return "Urbano (solar)"
    if has_industrial:
        return "Industrial"
    if has_rustico:
        return "Rústico"
    if has_urbanizable:
        return "Urbanizable"
    return pd.NA

df_tipo_suelo.loc[mask_infer, "tipo_suelo"] = df_tipo_suelo.loc[mask_infer, "notas"].apply(infer_tipo_suelo_from_notas)

# Estandarizar variantes residuales.
df_tipo_suelo["tipo_suelo"] = df_tipo_suelo["tipo_suelo"].replace({
    "Urbano (no consolidado)": "Urbanizable"
})

print(df_tipo_suelo["tipo_suelo"].value_counts(dropna=False))


tipo_suelo
Urbano (solar)    558
Urbanizable       364
No urbanizable    208
Urbano            120
<NA>               41
Industrial          5
Rústico             2
Mixto               1
Name: count, dtype: Int64


## Filtrado de nulos en tipo_suelo y eliminación de columnas irrelevantes

Tras agotar la inferencia desde `notas`, los registros con `tipo_suelo` nulo son irrecuperables y deben eliminarse — no tiene sentido imputar la tipología de suelo.

Se eliminan también columnas que ya cumplieron su función o tienen demasiados nulos para ser útiles.

In [ ]:
# ── Filtrar registros sin tipo_suelo (irrecuperables) ────────────────────
n_before = len(df_tipo_suelo)
df_tipo_suelo = df_tipo_suelo[df_tipo_suelo["tipo_suelo"].notna()].copy()
n_dropped = n_before - len(df_tipo_suelo)
print(f"Nulos en tipo_suelo eliminados: {n_dropped} ({100*n_dropped/n_before:.1f}%)")
print(f"Registros restantes: {len(df_tipo_suelo):,}")

# ── Eliminar columnas irrelevantes para el dataset final ──────────────────
# precio_anterior_eur / descuento_pct → ya capturadas en vendido_con_descuento
# referencia_catastral → 95% nulos, sin valor analítico
# notas → ya utilizada para inferir tipo_suelo, no aporta valor estructurado
# anunciante → 40% nulos, texto libre de alta cardinalidad
cols_drop = ["notas", "precio_anterior_eur", "descuento_pct", "referencia_catastral", "anunciante"]
cols_drop_present = [c for c in cols_drop if c in df_tipo_suelo.columns]
df_tipo_suelo = df_tipo_suelo.drop(columns=cols_drop_present)

print(f"\nColumnas eliminadas: {cols_drop_present}")
print(f"Columnas restantes: {df_tipo_suelo.columns.tolist()}")

Nulos en tipo_suelo eliminados: 41 (3.2%)
Registros restantes: 1,258

Columnas eliminadas: ['notas', 'precio_anterior_eur', 'descuento_pct', 'referencia_catastral', 'anunciante']
Columnas restantes: ['titulo', 'municipio', 'precio_eur', 'superficie_m2', 'tipo_suelo', 'vendido_con_descuento']


## Variable booleana: urbano o urbanizable

`es_urbano_o_urbanizable` marca `True` para suelos urbanos/urbanizables y `False` para el resto (incluyendo industriales y rústicos).


In [ ]:
df_urb = df_tipo_suelo.copy()

df_urb["es_urbano_o_urbanizable"] = df_urb["tipo_suelo"].isin(["Urbano (solar)", "Urbanizable"])

df_urb[["tipo_suelo", "es_urbano_o_urbanizable"]].head(10)


,tipo_suelo,es_urbano_o_urbanizable
id,,
2,Urbano (solar),True
3,Urbano (solar),True
4,Urbano (solar),True
5,Urbano (solar),True
6,Urbano (solar),True
7,Urbanizable,True
8,Urbano (solar),True
9,No urbanizable,False
10,Urbano (solar),True


## Precio por metro cuadrado

Se calcula `precio_m2 = precio_eur / superficie_m2`, evitando divisiones por cero y superficies no válidas.


In [ ]:
df_precio_m2 = df_urb.copy()

df_precio_m2.loc[df_precio_m2["superficie_m2"] <= 0, "superficie_m2"] = np.nan
df_precio_m2["precio_m2"] = df_precio_m2["precio_eur"] / df_precio_m2["superficie_m2"]

df_precio_m2[["precio_eur", "superficie_m2", "precio_m2"]].describe().T


,count,mean,std,min,25%,50%,75%,max
precio_eur,1258.0,263979.690779,380267.434637,1200.000000,80000.000000,149000.000000,292750.000000,5000000.0
superficie_m2,1258.0,6158.119237,11682.119251,1.000000,1295.000000,2832.500000,5731.250000,109758.0
precio_m2,1258.0,138.388263,1025.908888,0.610004,24.454688,61.970149,134.619547,36000.0


## Limpieza de título

Se elimina el prefijo `Terreno en ` al inicio del texto para estandarizar la variable `titulo`.


In [ ]:
df_titulo = df_precio_m2.copy()

df_titulo["titulo"] = (
    df_titulo["titulo"]
    .astype("string")
    .str.replace(r"^Terreno en\s+", "", regex=True)
    .str.strip()
)

df_titulo["titulo"].head(10)


id
2                                       Oruña, Piélagos
3                Peñacastillo - Nuevamontaña, Santander
4     Calle de la Arnia, Soto de la Marina, Santa Cr...
5                       Calle Barrio San Roque, Polanco
6     Avenida Playa San Juan de la Canal, Soto de la...
7                     Barrio la Venera, Oruña, Piélagos
8                          Barrio Mar, 87 --81, Polanco
9                                   Sv-4713, 12, Miengo
10             Barrio la Picota, 38 a, Renedo, Piélagos
11                   Los Pumares, 9, Santillana del Mar
Name: titulo, dtype: string

## Control final y exportación opcional

Se revisa el resultado final y se deja una línea opcional para guardar el dataset limpio.


In [ ]:
df_final = df_titulo.copy()

print("Shape final:", df_final.shape)
print("Nulos por columna:")
print(df_final.isna().sum().sort_values(ascending=False))

# Exportación opcional
# output_path = Path("..") / "data" / "1st_cleaning" / "terrenos_idealista_clean.csv"
# df_final.to_csv(output_path, sep=';', index=False)


Shape final: (1258, 8)
Nulos por columna:
titulo                     0
municipio                  0
precio_eur                 0
superficie_m2              0
tipo_suelo                 0
vendido_con_descuento      0
es_urbano_o_urbanizable    0
precio_m2                  0
dtype: int64


## Deduplicación

En datasets de scraping es frecuente que el mismo terreno aparezca bajo anuncios distintos. Se detectan duplicados por la combinación `titulo` + `municipio` + `precio_eur` + `superficie_m2`, conservando la primera aparición.

In [ ]:
# ── Deduplicación ─────────────────────────────────────────────────────────
dup_cols  = ["titulo", "municipio", "precio_eur", "superficie_m2"]
n_before  = len(df_final)
df_final  = df_final.drop_duplicates(subset=dup_cols, keep="first").reset_index(drop=True)
n_dropped = n_before - len(df_final)

print(f"Duplicados eliminados: {n_dropped} ({100*n_dropped/n_before:.1f}%)")
print(f"Registros restantes:   {len(df_final):,}")
print(f"Shape final:           {df_final.shape}")

Duplicados eliminados: 79 (6.3%)
Registros restantes:   1,179
Shape final:           (1179, 8)


In [ ]:
# EXPORT_PREPROCESSED_TERRENOS
from pathlib import Path

raw_base_candidates = [
    Path("data/raw"),
    Path("../../data/raw")
]
raw_base = next((p for p in raw_base_candidates if p.exists()), Path("data/raw"))

preprocessed_dir = raw_base / "scraping_manual" / "preprocessed"
preprocessed_dir.mkdir(parents=True, exist_ok=True)

output_path = preprocessed_dir / "scraping_land_preprocessed.csv"
df_final.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f"Dataset exportado en: {output_path.resolve()}")
print(f"Registros: {len(df_final):,}")

Dataset exportado en: C:\Users\pdelr\Documents\MBA\BezanillaSL\data\raw\scraping_manual\preprocessed\scraping_land_preprocessed.csv
Registros: 1,179


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df_out = df_final.copy()
cols_eda = ['precio_eur', 'superficie_m2', 'precio_m2']
labels_eda = ['Precio (EUR)', 'Superficie (m²)', 'Precio/m² (EUR/m²)']

# Distribuciones y boxplots
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribuciones – Terrenos (preprocesado)', fontsize=14, fontweight='bold')
for i, (col, label) in enumerate(zip(cols_eda, labels_eda)):
    data = df_out[col].dropna()
    axes[0, i].hist(data, bins=40, color='teal', edgecolor='white', linewidth=0.4)
    axes[0, i].set_title(label)
    axes[0, i].set_xlabel(label)
    axes[0, i].set_ylabel('Frecuencia')
    axes[1, i].boxplot(data, vert=False, patch_artist=True,
                       boxprops=dict(facecolor='teal', alpha=0.5))
    axes[1, i].set_xlabel(label)
    axes[1, i].set_yticks([])
plt.tight_layout()
plt.show()

# Q-Q plots + skewness
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Q-Q Plots – alejamiento de la normalidad', fontsize=12, fontweight='bold')
for ax, col, label in zip(axes, cols_eda, labels_eda):
    d = df_out[col].dropna()
    stats.probplot(d, dist='norm', plot=ax)
    ax.set_title(f"{label}\nskew={d.skew():.2f}  kurt={d.kurtosis():.2f}")
plt.tight_layout()
plt.show()

# Outliers por IQR
def mask_iqr(df, cols, factor=1.5):
    mask = pd.Series(False, index=df.index)
    for col in cols:
        d = df[col].dropna()
        q1, q3 = d.quantile(0.25), d.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - factor * iqr, q3 + factor * iqr
        mask |= df[col].notna() & ((df[col] < lower) | (df[col] > upper))
    return mask

mask_iqr15 = mask_iqr(df_out, cols_eda, 1.5)
mask_iqr30 = mask_iqr(df_out, cols_eda, 3.0)
print(f"Outliers IQR×1.5: {mask_iqr15.sum():,} ({100*mask_iqr15.mean():.1f}%)")
print(f"Outliers IQR×3.0: {mask_iqr30.sum():,} ({100*mask_iqr30.mean():.1f}%)")

# Isolation Forest (visión multivariante)
sub_if = df_out[cols_eda].copy().fillna(df_out[cols_eda].median())
clf = IsolationForest(contamination=0.02, random_state=42)
preds = clf.fit_predict(sub_if)
mask_if = preds == -1
print(f"Outliers IsolationForest (2%): {mask_if.sum():,} ({100*mask_if.mean():.1f}%)")

## Análisis de outliers (EDA)

Se realiza un análisis exploratorio de outliers sobre `precio_eur`, `superficie_m2` y `precio_m2` para entender la cola de la distribución antes de aplicar un pipeline de filtrado.